# Road Following — Interactive Regression (ROS Camera + Data Collection)

Collect data, train, and test the ResNet-18 model using **ROS Camera Topic** (`/csi_cam_0/image_raw`).

| Step | Description |
|------|-------------|
| 1 | Setup Environment & ROS Node |
| 2 | Configure Task & Dataset |
| 3 | Start ROS Camera Subscriber & Interactive Live Feed |
| 4 | Label & Save Data Samples via Python Cell |
| 5 | Setup Model Architecture (ResNet-18) |
| 6 | Train & Evaluate Model |
| 7 | Live Model Prediction Preview |

### 1. Setup Environment & ROS Node

In [ ]:
import os
import sys
import time
import cv2
import numpy as np
from pathlib import Path

# Add parent directory to sys.path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

import rospy
from sensor_msgs.msg import Image as ROSImage
try:
    from jetracer.utils import preprocess_onnx, bgr8_to_jpeg
except ImportError:
    from utils import preprocess_onnx, bgr8_to_jpeg

# Initialize ROS Node
try:
    rospy.init_node('interactive_regression_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")


### 2. Configure Task & Dataset

In [ ]:
from xy_dataset import XYDataset

TASK = 'road_following'
CATEGORIES = ['apex']
DATASETS = ['A', 'B']

datasets = {}
for name in DATASETS:
    datasets[name] = XYDataset(TASK + '_' + name, CATEGORIES, random_hflip=True)

dataset_name = DATASETS[0]
dataset = datasets[dataset_name]
category = CATEGORIES[0]

print(f"[*] Task: {TASK}")
print(f"[*] Categories: {CATEGORIES}")
print(f"[*] Active Dataset: {dataset_name} ({len(dataset)} samples)")


### 3. Start ROS Camera Stream & Interactive Labeling Interface

Click on the image below (or run Step 4 cell) to save a labeled data sample at coordinate (x, y).

In [ ]:
import base64
import threading
from IPython.display import display, HTML

latest_image = None
latest_jpeg = None
last_clicked_pt = None
_lock = threading.Lock()

target_x = 112
target_y = 180

# Function called to save labeled sample
def collect_sample(x=None, y=None):
    global latest_image, dataset, category, last_clicked_pt, target_x, target_y
    if latest_image is None:
        print("[!] ERROR: No camera frame received from ROS yet!")
        return
    
    if x is not None: target_x = int(max(0, min(224, x)))
    if y is not None: target_y = int(max(0, min(224, y)))
    last_clicked_pt = (target_x, target_y)

    dataset.save_entry(category, latest_image, target_x, target_y)
    count = len(dataset)
    print(f"[+] Labeled & Saved Sample #{count} -> Category: '{category}', (X={target_x}, Y={target_y})")

def save_clicked_sample(x, y):
    collect_sample(x, y)

sys.modules['__main__'].save_clicked_sample = save_clicked_sample
sys.modules['__main__'].collect_sample = collect_sample

# IPython Display Handle
dh = display(HTML("<p><b>Waiting for ROS Camera Feed...</b></p>"), display_id=True)

def ros_image_to_cv2(msg):
    im = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, -1)
    if msg.encoding in ['rgb8', 'rgb8']:
        im = cv2.cvtColor(im, cv2.COLOR_RGB2BGR)
    elif msg.encoding == 'rgba8':
        im = cv2.cvtColor(im, cv2.COLOR_RGBA2BGR)
    elif msg.encoding == 'bgra8':
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    return im

def camera_callback(msg):
    global latest_image, latest_jpeg
    acquired = _lock.acquire(blocking=False)
    if not acquired:
        return
    try:
        cv_img = ros_image_to_cv2(msg)
        latest_image = cv_img.copy()
        
        preview = cv_img.copy()
        h, w = preview.shape[:2]
        
        # Draw last labeled point or current target point
        pt = last_clicked_pt if last_clicked_pt is not None else (target_x, target_y)
        cv2.circle(preview, pt, 8, (0, 255, 0), -1)
        cv2.putText(preview, f"Target X:{pt[0]} Y:{pt[1]}", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        jpeg_bytes = bgr8_to_jpeg(preview)
        latest_jpeg = jpeg_bytes
        
        b64 = base64.b64encode(jpeg_bytes).decode('utf-8')
        count = len(dataset)
        
        html_content = f'''
        <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 12px; border-radius: 8px; display: inline-block;">
            <h4 style="margin:0 0 6px 0; color: #ffffff;">ROS Camera Feed (/csi_cam_0/image_raw)</h4>
            <p style="margin:4px 0;"><b>Dataset:</b> {dataset_name} | <b>Category:</b> {category} | <b>Total Samples:</b> {count}</p>
            <p style="margin:4px 0; color: #ffff00;"><b>Current Target Point:</b> X={pt[0]}, Y={pt[1]}</p>
            
            <div style="position: relative; display: inline-block; cursor: crosshair;">
                <img id="label_img" src="data:image/jpeg;base64,{b64}" 
                     style="width:224px; height:224px; border:2px solid #00ff00; border-radius:4px; display:block;" 
                     onclick="onImageClick(event)" />
            </div>

            <script>
            function onImageClick(event) {{
                var img = document.getElementById('label_img');
                var rect = img.getBoundingClientRect();
                var scaleX = 224 / rect.width;
                var scaleY = 224 / rect.height;
                var x = Math.round((event.clientX - rect.left) * scaleX);
                var y = Math.round((event.clientY - rect.top) * scaleY);
                var code = 'save_clicked_sample(' + x + ', ' + y + ')';

                if (window.Jupyter && Jupyter.notebook && Jupyter.notebook.kernel) {{
                    Jupyter.notebook.kernel.execute(code);
                }} else if (window.google && google.colab) {{
                    google.colab.kernel.invokeFunction('save_clicked_sample', [x, y], {{}});
                }}
            }}
            </script>
        </div>
        '''
        dh.update(HTML(html_content))
    except Exception as e:
        print(f"[!] Callback exception: {e}")
    finally:
        _lock.release()

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, camera_callback, queue_size=1, buff_size=2**24)
print(f"[*] Subscribed to ROS Camera Topic: {topic_name}")

published_topics = [t[0] for t in rospy.get_published_topics()]
if topic_name not in published_topics:
    print(f"\n[!] WARNING: Topic '{topic_name}' is NOT actively publishing!")
    print(f"    Make sure to run 'bash launch_camera.sh' in Terminal 1 on Jetson Nano.")
else:
    print(f"[+] Topic '{topic_name}' is ACTIVE!")


### 4. Label & Save Data Samples via Python Cell

In [ ]:
# Run this cell to label and save the current camera frame!
# Set (x, y) coordinates for target point on the 224x224 image:
# X: 0 (left) -> 112 (center) -> 224 (right)
# Y: 0 (top)  -> 112 (center) -> 224 (bottom/hood)

collect_sample(x=112, y=180)


### 5. Setup Model Architecture (ResNet-18)

In [ ]:
import torch
import torchvision

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dim = 2 * len(dataset.categories)  # x, y coordinate for category

# ResNet-18
model = torchvision.models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, output_dim)
model = model.to(device)

model_path = 'road_following_model.pth'

def save_model(path=model_path):
    torch.save(model.state_dict(), path)
    print(f"[+] Model saved to '{path}'")

def load_model(path=model_path):
    if os.path.exists(path):
        model.load_state_dict(torch.load(path))
        print(f"[+] Model loaded from '{path}'")
    else:
        print(f"[!] Model file '{path}' not found.")

print(f"[*] ResNet-18 Model initialized on device: {device}")


### 6. Train & Evaluate Model

In [ ]:
def train_model(epochs=5, batch_size=8, lr=1e-3):
    if len(dataset) == 0:
        print("[!] Dataset is empty. Collect data samples in Step 4 first!")
        return

    train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    print(f"[*] Starting Training for {epochs} epochs (Total samples: {len(dataset)})...")
    
    for epoch in range(epochs):
        sum_loss = 0.0
        count = 0
        for images, category_idx, xy in iter(train_loader):
            images = images.to(device)
            xy = xy.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = 0.0
            for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx + 2] - xy[batch_idx]) ** 2)
            loss /= len(category_idx)

            loss.backward()
            optimizer.step()

            sum_loss += float(loss) * len(category_idx)
            count += len(category_idx)

        avg_loss = sum_loss / count
        print(f"  Epoch [{epoch+1}/{epochs}] - Training Loss: {avg_loss:.4f}")

    model.eval()
    save_model(model_path)
    print("[+] Training completed & model saved successfully!")

# Run line below to start training:
# train_model(epochs=5)


### 7. Live Model Prediction Preview (Red Target Point Overlay)

In [ ]:
# Run this cell to PREVIEW model predictions on live camera feed
# Interrupt/stop kernel to stop live preview.

predict_dh = display(HTML("<p><b>Waiting for Live Prediction...</b></p>"), display_id=True)
live_running = True

print("[+] Starting Model Prediction Live Preview... Interrupt kernel cell to stop.")

try:
    model.eval()
    while live_running:
        if latest_image is not None:
            input_tensor = preprocess_onnx(latest_image)
            with torch.no_grad():
                img_t = torch.from_numpy(input_tensor).to(device)
                outputs = model(img_t).cpu().numpy().flatten()

            raw_x = float(outputs[0])
            raw_y = float(outputs[1]) if len(outputs) > 1 else 0.0

            h, w = latest_image.shape[:2]
            px = int(w * (raw_x / 2.0 + 0.5))
            py = int(h * (raw_y / 2.0 + 0.5)) if raw_y != 0.0 else int(h * 0.5)

            pred_frame = latest_image.copy()
            cv2.circle(pred_frame, (px, py), 8, (0, 0, 255), -1)
            cv2.putText(pred_frame, f"Pred X:{raw_x:+.2f} Y:{raw_y:+.2f}", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            jpeg_bytes = bgr8_to_jpeg(pred_frame)
            b64 = base64.b64encode(jpeg_bytes).decode('utf-8')

            html = f'''
            <div style="font-family: monospace; background: #1e1e1e; color: #ff5555; padding: 12px; border-radius: 8px; display: inline-block;">
                <h4 style="margin:0 0 6px 0; color: #ffffff;">Model Prediction Live Preview</h4>
                <p style="margin:4px 0;"><b>Predicted X:</b> {raw_x:+.3f} | <b>Predicted Y:</b> {raw_y:+.3f}</p>
                <img src="data:image/jpeg;base64,{b64}" style="width:320px; height:auto; border:2px solid #ff5555; border-radius:4px; margin-top:6px;" />
            </div>
            '''
            predict_dh.update(HTML(html))
        time.sleep(0.05)
except KeyboardInterrupt:
    live_running = False
    print("[+] Live preview stopped.")
